# GDELT Preprocessing Pipeline — Raw Signals to Reduced Relevant Corpus

Implements **Step 1** of the Risk Agent build plan: takes every `gdelt_*.csv` file in a
directory (any number of months), and produces a single deduplicated, relevance-filtered
**story-level corpus**, where each row is a unique news story with a full list of every
raw file/row location it appeared in.

**Why this matters (confirmed against the actual data):**
- Only ~8-12% of raw GDELT rows are relevant to energy/shipping/geopolitical risk — the rest
  is domestic news (the `india` corridor tag is ~87% unrelated coverage: school admissions,
  court cases, cricket, etc).
- The same story is often duplicated dozens to hundreds of times as separate rows (GDELT logs
  multiple "events" per article), and **the same story can also appear across different
  monthly files** if it was scraped near a month boundary or re-syndicated later — so
  deduplication must happen *after* concatenating every file, not per-file.
- There is no article full-text in this data — only URL, entities, and a bare severity score.
  All relevance filtering here works off URL slug + entity tokens, which is a **proxy**, not
  true semantic filtering. That limitation is called out explicitly where it matters, with a
  clear upgrade path (Stage B is written as a drop-in swap for real embeddings once article
  text is fetched).

**Pipeline stages:**
1. Load & concatenate all monthly files, tagging every row with its exact source location.
2. Normalize URLs and group rows into **stories** (cross-file, cross-row dedup).
3. Stage A — keyword filter (cheap, deterministic).
4. Stage B — TF-IDF semantic similarity filter, calibrated against Stage A as an anchor,
   to rescue paraphrased/non-keyword stories that are still genuinely relevant.
5. Corridor reassignment — correct GDELT's coarse corridor tag using matched location terms.
6. Assemble the final reduced corpus with full location provenance.
7. QC funnel — verify every stage's row counts, so the reduction is auditable, not a black box.
8. Export to CSV (flattened) and JSON (full fidelity, nested locations).


## 0. Config — check and adjust these before running

In [1]:
import glob, re, json, hashlib
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- INPUT ---
# Directory containing gdelt_*.csv files (any number of months). All files matching
# the pattern are loaded and deduplicated together, so cross-file duplicate stories
# are caught correctly.
INPUT_DIR = "D:\\python time\\ET\\code\\risk_agent\\csv_backfill\\gdelt"
FILE_PATTERN = "gdelt_*.csv"

# --- OUTPUT ---
OUTPUT_DIR = "D:\\python time\\ET\\code\\risk_agent\\csv_backfill\\gdelt\\outputs"
OUTPUT_CSV = f"{OUTPUT_DIR}/gdelt_relevant_corpus.csv"
OUTPUT_JSON = f"{OUTPUT_DIR}/gdelt_relevant_corpus.json"

# --- STAGE A: keyword filter ---
# Two tiers, not one flat list. This matters for two reasons, both confirmed against the
# actual data:
#
# 1. WORD BOUNDARIES: a flat substring search (no ) matches keywords *inside* unrelated
#    words — e.g. "port" was matching inside "report", "reportage", "important", producing
#    ~24,000 false positives on its own. All matching below uses  (whole-word) regex.
#
# 2. ANCHOR vs CONTEXT: broad war/conflict/military words (attack, strike, military, war,
#    coup...) are far too common in unrelated domestic news to stand alone as keywords —
#    tested directly: ~23,700 rows contain a bare context word with no relevant place/actor
#    attached (protests, court cases, unrelated global military news). So context words only
#    count as a match when they co-occur with an ANCHOR term (a place, actor, or
#    energy/shipping-specific word) in the same story. Anchor terms are specific enough to
#    stand alone. This is what actually lets you safely add the war/conflict category without
#    reflooding the noise the earlier filtering removed.
ANCHOR_KEYWORDS = [
    "oil", "crude", "tanker", "opec", "opec+", "hormuz", "strait", "sanction", "embargo",
    "chokepoint", "refin", "pipeline", "lng", "houthi", "red sea", "red-sea",
    "bab el mandeb", "bab-el-mandeb", "persian gulf", "gulf", "iran", "yemen",
    "vessel", "shipping", "petrol", "fuel", "energy", "port",
]
CONTEXT_KEYWORDS = [
    "war", "military", "conflict", "invasion", "ceasefire", "mobiliz", "naval", "warship",
    "missile", "drone", "strike", "attack", "coup", "insurgency", "escalation",
    "retaliation", "troops", "blockade", "martial law", "curfew", "seize", "seized",
]

# --- STAGE B: TF-IDF semantic rescue ---
# Hand-written seed sentences describing genuine disruption events. Stories whose pseudo-text
# (URL slug + entities) is similar to this seed set get rescued even if no exact keyword hit.
SEED_SENTENCES = [
    "tanker attacked near strait",
    "vessel seized by naval forces",
    "port suspends operations due to conflict",
    "opec announces surprise production cut",
    "oil prices spike after supply disruption",
    "shipping company suspends transit through red sea",
    "houthi drone strike on cargo ship",
    "sanctions imposed on oil exports",
    "refinery shutdown after strike",
    "strait of hormuz blockade threat",
]

# Stage B threshold is calibrated automatically below (median non-zero TF-IDF score among
# Stage-A-matched stories) rather than hardcoded, so it re-calibrates itself as more months
# of data are added. You can override it manually after reviewing the QC cell.
STAGE_B_THRESHOLD_OVERRIDE = None  # e.g. set to 0.05 to override the auto-calibrated value

# --- CORRIDOR REASSIGNMENT ---
# GDELT's own corridor_id tag is coarse (esp. the "india" bucket). If location terms for a
# specific corridor appear in a story's text, reassign it to that corridor.
CORRIDOR_KEYWORDS = {
    "hormuz": ["hormuz"],
    "bab_el_mandeb": ["bab el mandeb", "bab-el-mandeb", "bab al mandab"],
    "suez": ["suez"],
    "red_sea": ["red sea", "red-sea"],
    "persian_gulf": ["persian gulf", "arabian gulf"],
}


## 1. Load & concatenate every monthly file

Every row is tagged with `source_file` and `source_row` — this is the "location" that survives
all the way to the final output, so you can always trace a relevant story back to its exact
raw file and row.

In [2]:
files = sorted(glob.glob(f"{INPUT_DIR}/{FILE_PATTERN}"))
assert files, f"No files matched {INPUT_DIR}/{FILE_PATTERN}"
print(f"Found {len(files)} file(s):")
for f in files:
    print(" -", f.split('/')[-1])

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["source_file"] = f.split("/")[-1]
    df["source_row"] = df.index
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal raw rows across all files: {len(raw):,}")
raw.head(3)


Found 13 file(s):
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-07.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-08.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-09.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-10.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-11.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2025-12.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-01.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-02.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-03.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-04.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-05.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-06.csv
 - D:\python time\ET\code\risk_agent\csv_backfill\gdelt\gdelt_2026-07.csv

Total raw rows acro

,signal_id,corridor_id,as_of,signal_type,metric,value,unit,data_status,confidence,source,generated_at,raw_reference,entities,dedup_key,source_file,source_row
0,gdelt:india:event_severity:2025-07-22T10:15:00...,india,2025-07-22T10:15:00+00:00,NEWS_EVENT,event_severity,0.449379,score_0_1,OBSERVED,0.1,gdelt,2026-07-18T16:24:33.885455+00:00,https://www.prokerala.com/news/articles/a16557...,CONGRESS|FARMER,1253358754,D:\python time\ET\code\risk_agent\csv_backfill...,0
1,gdelt:india:event_severity:2025-08-13T14:00:00...,india,2025-08-13T14:00:00+00:00,NEWS_EVENT,event_severity,0.716462,score_0_1,OBSERVED,0.1,gdelt,2026-07-18T16:24:33.885455+00:00,https://www.rediff.com/news/report/bombay-plan...,|AUTHORITIES,1257309872,D:\python time\ET\code\risk_agent\csv_backfill...,1
2,gdelt:india:event_severity:2026-07-09T01:45:00...,india,2026-07-09T01:45:00+00:00,NEWS_EVENT,event_severity,0.358813,score_0_1,OBSERVED,0.1,gdelt,2026-07-18T16:24:33.885455+00:00,https://kashmirreader.com/2026/07/09/mosque-pr...,THE OPPOSITION|,1312748456,D:\python time\ET\code\risk_agent\csv_backfill...,2


## 2. Normalize URLs and build story IDs (cross-file dedup key)

URL normalization strips scheme, `www.`, trailing slash, query string, and fragment, so the
same article reached via slightly different links (tracking params, http vs https, etc.)
collapses to one story. `story_id` is a stable short hash of the normalized URL — this is the
join key used everywhere downstream, **and it's computed after concatenating every file**, so
duplicates across different monthly files are caught correctly, not just within a file.

In [3]:
# Query strings are NOT dropped wholesale: many sites (esp. WordPress-style CMSs seen in
# this data, e.g. navabharat.com/?p=258318) encode the actual article ID in the query string,
# not the path. Dropping the query string entirely was tested and caused a real bug: it
# collapsed hundreds of genuinely distinct articles from the same domain into one fake
# "mega-story" at the domain root. Instead, only known tracking parameters are stripped;
# everything else (including CMS article IDs) is kept and sorted for stability.
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "fbclid", "gclid", "ref", "spm", "cmpid",
}

def normalize_url(u):
    if not isinstance(u, str) or not u.strip():
        return None
    try:
        p = urlparse(u.strip())
        netloc = p.netloc.lower()
        if netloc.startswith("www."):
            netloc = netloc[4:]
        path = p.path.rstrip("/")
        qs = [(k, v) for k, v in parse_qsl(p.query) if k.lower() not in TRACKING_PARAMS]
        qs.sort()
        query = urlencode(qs)
        return urlunparse(("", netloc, path, "", query, ""))
    except Exception:
        return u.strip().lower()

def make_story_id(url, fallback_key):
    # A "bare" reference (no path, no query, no fragment - literally just a domain) is
    # ambiguous: GDELT couldn't resolve a specific article URL for that event. Grouping
    # rows by an identical bare URL is technically correct URL-matching, but semantically
    # wrong — it silently merges unrelated events under one fake shared "story", the same
    # class of bug as the earlier query-string issue, just from a different root cause.
    # Confirmed present in later months' data via the QC cell below. Fix: for bare/ambiguous
    # references, don't dedupe by URL at all — fall back to a per-row unique key so each
    # event stays its own story instead of merging into a fake mega-cluster.
    p = urlparse(url)
    is_bare = p.path.rstrip("/") == "" and p.query == "" and p.fragment == ""
    if is_bare:
        return hashlib.sha1(f"unresolved::{fallback_key}".encode("utf-8")).hexdigest()[:16]
    return hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]

raw["canonical_url"] = raw["raw_reference"].apply(normalize_url)
raw["domain"] = raw["raw_reference"].apply(
    lambda u: urlparse(u).netloc.lower().replace("www.", "") if isinstance(u, str) else None
)

def _is_bare(url):
    if not url:
        return True
    p = urlparse(url)
    return p.path.rstrip("/") == "" and p.query == "" and p.fragment == ""

raw["is_ambiguous_reference"] = raw["canonical_url"].apply(_is_bare)
raw["story_id"] = raw.apply(
    lambda r: make_story_id(r["canonical_url"], r["signal_id"]) if r["canonical_url"] else None,
    axis=1,
)

n_ambiguous = int(raw["is_ambiguous_reference"].sum())
if n_ambiguous:
    print(f"NOTE: {n_ambiguous:,} rows have an ambiguous (bare-domain) source reference — "
          f"each is kept as its own separate story rather than merged by URL.")

n_null_url = raw["canonical_url"].isna().sum()
n_unique_stories = raw["story_id"].nunique()
print(f"Rows with no usable URL (dropped from story grouping): {n_null_url:,}")
print(f"Raw rows: {len(raw):,}  ->  Unique stories after cross-file dedup: {n_unique_stories:,}")

mention_counts = raw.groupby("story_id").size().sort_values(ascending=False)
print("\nMost-duplicated stories (top 5 — sanity check these look like real repeat coverage,")
print("not a normalization bug):")
print(mention_counts.head(5))


NOTE: 80 rows have an ambiguous (bare-domain) source reference — each is kept as its own separate story rather than merged by URL.
Rows with no usable URL (dropped from story grouping): 0
Raw rows: 4,863,286  ->  Unique stories after cross-file dedup: 1,520,843

Most-duplicated stories (top 5 — sanity check these look like real repeat coverage,
not a normalization bug):
story_id
6624cf31aab44c48    170
140d8ca2b91cc171    168
51c1259b30658f46    167
ae204c1b5aa33cd3    165
b21f20eba141f20c    165
dtype: int64


**Why this check matters:** two separate bugs of the same class were found and fixed here
while testing against real data:
1. An earlier version stripped query strings entirely, which collapsed distinct articles into
   one fake "mega-story" whenever a site encoded its article ID in the query string (e.g.
   WordPress-style `?p=12345` URLs).
2. A later run against the full 13-month backfill found that some sources provide genuinely
   bare-domain references (GDELT couldn't resolve a specific article URL for that event). These
   are now treated as ambiguous and kept as separate singleton stories (`is_ambiguous_reference`)
   rather than merged by URL.

The first check below verifies fix #2 held structurally (should always be 0 by construction).
The second is a general outlier check — a legitimately popular, genuinely-repeated story can
have a high mention count, so this isn't "should be 0," it's "should be manually eyeballed" —
if a top entry doesn't look like real repeat coverage when you check its URL, that's a sign a
new URL-scheme edge case has appeared and needs the same treatment as #1 and #2.

In [4]:
# Check 1: ambiguous references should never end up grouped with >1 mention post-fix
ambiguous_grouped = raw[raw["is_ambiguous_reference"]].groupby("story_id").size()
bad_ambiguous = ambiguous_grouped[ambiguous_grouped > 1]
print(f"Ambiguous references incorrectly grouped together: {len(bad_ambiguous)} (should be 0)")

# Check 2: general outlier flag on genuinely-resolved (non-ambiguous) URLs, for manual review
resolved = raw[~raw["is_ambiguous_reference"]]
mention_counts_resolved = resolved.groupby("story_id").size().sort_values(ascending=False)
q99 = mention_counts_resolved.quantile(0.99)
print(f"\n99th percentile mention count among resolved URLs: {q99:.0f}")
print("Top 5 by mention count — manually check these URLs look like real repeat coverage:")
top5 = mention_counts_resolved.head(5)
for sid, cnt in top5.items():
    sample_url = raw[raw["story_id"] == sid]["canonical_url"].iloc[0]
    print(f"  {cnt:>4} mentions  ->  {sample_url}")


Ambiguous references incorrectly grouped together: 0 (should be 0)

99th percentile mention count among resolved URLs: 18
Top 5 by mention count — manually check these URLs look like real repeat coverage:
   170 mentions  ->  //news18.com/india/lok-sabha-speaker-om-birla-to-represent-india-at-bangladesh-pm-elect-tarique-rahmans-swearing-in-9905123.html
   168 mentions  ->  //heritagefl.com/story/2025/10/03/news/trump-presents-gaza-peace-plan-to-muslim-leaders-at-un/22253.html
   167 mentions  ->  //republicworld.com/india/mediation-a-game-changer-for-faster-off-court-settlements-justice-surya-kant-takes-oath-as-53rd-cji
   165 mentions  ->  //upi.com/Top_News/World-News/2026/01/22/nine-Muslim-countries-join-board-of-peace/4621769067535
   165 mentions  ->  //thehill.com/homenews/administration/5529126-donald-trump-nobel-peace-prize-insult


## 3. Build pseudo-text and locations per row

There's no article body text in this data, so relevance filtering works off a **pseudo-text**
field: URL slug tokens + the `entities` column. This is a proxy — good enough for keyword and
coarse TF-IDF filtering, not a substitute for filtering on real article text. If you later fetch
full text from `raw_reference`, swap it in here and both Stage A and Stage B improve for free.

In [5]:
def slug_text(url):
    if not url:
        return ""
    path = url.split("/", 1)[-1] if "/" in url else ""
    return re.sub(r"[-_/]+", " ", path)

raw["slug_text"] = raw["canonical_url"].apply(slug_text)
raw["pseudo_text"] = (raw["slug_text"].fillna("") + " " + raw["entities"].fillna("")).str.lower()

# Per-row location record — this is what gets preserved into the final corpus's `locations` list.
raw["location"] = raw.apply(
    lambda r: {
        "source_file": r["source_file"],
        "source_row": int(r["source_row"]),
        "signal_id": r["signal_id"],
        "corridor_id": r["corridor_id"],
        "as_of": r["as_of"],
        "confidence": r["confidence"],
    },
    axis=1,
)
raw[["canonical_url", "pseudo_text"]].head(3)


,canonical_url,pseudo_text
0,//prokerala.com/news/articles/a1655779.html,prokerala.com news articles a1655779.html con...
1,//rediff.com/news/report/bombay-plans-to-allow...,rediff.com news report bombay plans to allow ...
2,//kashmirreader.com/2026/07/09/mosque-promised...,kashmirreader.com 2026 07 09 mosque promised ...


## 4. Stage A — keyword filter (anchor + context, whole-word matching)

Cheap, deterministic, no model calls. A story matches if it contains **any anchor term**, or
**a context term co-occurring with an anchor term** in the same story's pseudo-text. All
matching uses `\b` word boundaries — a naive substring search was tested and found to match
"port" inside "report"/"important" etc., producing ~24,000 false positives on that term alone.

In [6]:
anchor_pattern = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in ANCHOR_KEYWORDS) + r")\b", re.IGNORECASE
)
context_pattern = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in CONTEXT_KEYWORDS) + r")\b", re.IGNORECASE
)

raw["anchor_hits"] = raw["pseudo_text"].apply(lambda t: sorted(set(anchor_pattern.findall(t))))
raw["context_hits"] = raw["pseudo_text"].apply(lambda t: sorted(set(context_pattern.findall(t))))
raw["has_anchor"] = raw["anchor_hits"].apply(len).gt(0)
raw["has_context"] = raw["context_hits"].apply(len).gt(0)

# A row matches Stage A if it has an anchor term, OR a context term co-occurring with an
# anchor term. Context terms alone never qualify — that's the noise-control mechanism.
raw["keyword_matched"] = raw["has_anchor"]
raw["keyword_hits"] = raw.apply(lambda r: sorted(set(r["anchor_hits"] + (r["context_hits"] if r["has_anchor"] else []))), axis=1)

n_anchor_only = int((raw["has_anchor"] & ~raw["has_context"]).sum())
n_anchor_and_context = int((raw["has_anchor"] & raw["has_context"]).sum())
n_context_excluded = int((raw["has_context"] & ~raw["has_anchor"]).sum())

print(f"Stage A: {raw['keyword_matched'].sum():,} / {len(raw):,} raw rows keyword-matched "
      f"({raw['keyword_matched'].mean():.1%})")
print(f"  - anchor term only: {n_anchor_only:,}")
print(f"  - anchor + war/conflict context term together: {n_anchor_and_context:,}")
print(f"  - bare context term with no anchor (correctly excluded as noise): {n_context_excluded:,}")


Stage A: 402,856 / 4,863,286 raw rows keyword-matched (8.3%)
  - anchor term only: 290,779
  - anchor + war/conflict context term together: 112,077
  - bare context term with no anchor (correctly excluded as noise): 245,928


## 5. Stage B — TF-IDF semantic rescue (offline proxy for embeddings)

This environment has no network access to an embeddings API/model, so Stage B uses TF-IDF +
cosine similarity against the seed sentences as an **offline proxy for semantic filtering**.
It's weaker than real embeddings (short, sparse pseudo-text means most scores are 0), so its
job here is narrow and specific: **rescue a small number of high-similarity stories that Stage A's
exact keyword match missed** (e.g. paraphrased coverage) — not to replace Stage A.

The threshold is auto-calibrated as the median *non-zero* TF-IDF score among Stage-A-matched
stories, i.e. "how similar does a non-keyword story need to be to look as relevant as a typical
keyword-matched one." Recalibrate this each time you add more months, since vocabulary size
changes the score distribution.

**Upgrade path:** once you fetch full article text (or add a real embeddings model — the sandbox
here can't reach Hugging Face/OpenAI, but your own environment likely can), replace
`pseudo_text` with the real article text and swap `TfidfVectorizer` for a sentence-embedding
model. Everything downstream is unchanged.

In [7]:
# Aggregate pseudo-text to story level (union of all row-level tokens for that story)
story_text = (
    raw.groupby("story_id")["pseudo_text"]
    .apply(lambda s: " ".join(sorted(set(" ".join(s).split()))))
    .reset_index()
)

corpus = story_text["pseudo_text"].tolist() + SEED_SENTENCES
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X = vectorizer.fit_transform(corpus)

story_vecs = X[: len(story_text)]
seed_vecs = X[len(story_text):]
seed_centroid = np.asarray(seed_vecs.mean(axis=0))
story_text["tfidf_score"] = cosine_similarity(story_vecs, seed_centroid).flatten()

story_kw = raw.groupby("story_id")["keyword_matched"].any().reset_index()
story_text = story_text.merge(story_kw, on="story_id")

nonzero_kw_scores = story_text.loc[
    (story_text["keyword_matched"]) & (story_text["tfidf_score"] > 0), "tfidf_score"
]
auto_threshold = float(nonzero_kw_scores.median()) if len(nonzero_kw_scores) else 0.05
stage_b_threshold = STAGE_B_THRESHOLD_OVERRIDE or auto_threshold
print(f"Auto-calibrated Stage B threshold: {auto_threshold:.4f}")
print(f"Using threshold: {stage_b_threshold:.4f}")

rescued_mask = (~story_text["keyword_matched"]) & (story_text["tfidf_score"] >= stage_b_threshold)
print(f"Stage B rescued {rescued_mask.sum():,} additional stories "
      f"(out of {(~story_text['keyword_matched']).sum():,} non-keyword stories)")

story_text["stage_b_rescued"] = rescued_mask


Auto-calibrated Stage B threshold: 0.0602
Using threshold: 0.0602
Stage B rescued 10,187 additional stories (out of 1,379,108 non-keyword stories)


## 6. Corridor reassignment

GDELT's own `corridor_id` tag is coarse — the `india` bucket in particular sweeps in anything
tagged India regardless of relevance. If a story's pseudo-text contains a specific corridor's
location terms, reassign it. This is a simple, transparent heuristic — reasonable for a
prototype, not a substitute for a real entity/geo-resolution model at scale.

In [8]:
story_pseudo = (
    raw.groupby("story_id")["pseudo_text"]
    .apply(lambda s: " ".join(s))
    .reset_index()
    .rename(columns={"pseudo_text": "story_pseudo_text_full"})
)
story_orig_corridor = (
    raw.groupby("story_id")["corridor_id"].agg(lambda s: s.mode().iat[0]).reset_index()
)

corridors = story_orig_corridor.merge(story_pseudo, on="story_id")

def reassign_corridor(text, original):
    for corridor, kws in CORRIDOR_KEYWORDS.items():
        for kw in kws:
            if kw in text:
                return corridor
    return original

corridors["corridor_final"] = corridors.apply(
    lambda r: reassign_corridor(r["story_pseudo_text_full"], r["corridor_id"]), axis=1
)
n_reassigned = (corridors["corridor_final"] != corridors["corridor_id"]).sum()
print(f"Reassigned corridor for {n_reassigned:,} stories")
print(
    corridors[corridors["corridor_final"] != corridors["corridor_id"]]
    .groupby(["corridor_id", "corridor_final"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)


Reassigned corridor for 10,469 stories
corridor_id    corridor_final
india          hormuz            4868
persian_gulf   hormuz            3704
red_sea        hormuz             557
india          persian_gulf       346
red_sea        suez               258
hormuz         persian_gulf       202
india          red_sea             83
red_sea        persian_gulf        64
bab_el_mandeb  hormuz              56
persian_gulf   red_sea             49
dtype: int64


## 7. Assemble the final relevant corpus

A story is kept if it passed Stage A **or** was rescued by Stage B. For each kept story:
locations (every raw file/row it appeared in), unioned entities, mention count, first-seen
timestamp, and the final corridor.

In [9]:
story_text = story_text.merge(corridors[["story_id", "corridor_final"]], on="story_id")

url_lookup = raw.drop_duplicates("story_id").set_index("story_id")["canonical_url"]
domain_lookup = raw.drop_duplicates("story_id").set_index("story_id")["domain"]

locations_by_story = raw.groupby("story_id")["location"].apply(list)
entities_by_story = raw.groupby("story_id")["entities"].apply(
    lambda s: sorted(set(x for x in s.dropna() if x))
)
first_seen_by_story = raw.groupby("story_id")["as_of"].min()
all_keyword_hits_by_story = raw.groupby("story_id")["keyword_hits"].apply(
    lambda lists: sorted(set(h for hits in lists for h in hits))
)

story_text["is_relevant"] = story_text["keyword_matched"] | story_text["stage_b_rescued"]

corpus = story_text[story_text["is_relevant"]].copy()
corpus["canonical_url"] = corpus["story_id"].map(url_lookup)
corpus["domain"] = corpus["story_id"].map(domain_lookup)
corpus["first_seen_as_of"] = corpus["story_id"].map(first_seen_by_story)
corpus["mention_count"] = corpus["story_id"].map(locations_by_story.apply(len))
corpus["entities"] = corpus["story_id"].map(entities_by_story)
corpus["keyword_hits"] = corpus["story_id"].map(all_keyword_hits_by_story)
corpus["locations"] = corpus["story_id"].map(locations_by_story)
corpus["relevance_reason"] = np.where(
    corpus["keyword_matched"], "keyword_match", "stage_b_semantic_rescue"
)

corpus = corpus.rename(columns={"corridor_final": "corridor_id"})
corpus = corpus[
    [
        "story_id", "canonical_url", "domain", "corridor_id", "first_seen_as_of",
        "mention_count", "relevance_reason", "keyword_hits", "tfidf_score",
        "entities", "locations",
    ]
].sort_values("first_seen_as_of").reset_index(drop=True)

print(f"Final relevant corpus: {len(corpus):,} unique stories")
corpus.head(3)


Final relevant corpus: 151,922 unique stories


,story_id,canonical_url,domain,corridor_id,first_seen_as_of,mention_count,relevance_reason,keyword_hits,tfidf_score,entities,locations
0,20f9ec06efd268f4,//tbsnews.net/worldbiz/middle-east/iran-made-p...,tbsnews.net,hormuz,2025-07-02T02:30:00+00:00,1,keyword_match,"[hormuz, iran, strait]",0.122401,[US OFFICIAL|IRAN],[{'source_file': 'D:\python time\ET\code\risk_...
1,01d9e122f614f50a,//aol.co.uk/trump-claims-israel-ready-gaza-020...,aol.co.uk,persian_gulf,2025-07-02T03:00:00+00:00,2,keyword_match,[iran],0.000000,"[IRAN|ISRAEL, ISRAEL|IRAN]",[{'source_file': 'D:\python time\ET\code\risk_...
2,b5e53845b8e79368,//thedailystar.net/news/bangladesh/news/kuet-s...,thedailystar.net,india,2025-07-02T03:30:00+00:00,1,stage_b_semantic_rescue,[],0.109517,[EDUCATION MINIST|],[{'source_file': 'D:\python time\ET\code\risk_...


## 8. QC funnel — understand exactly what each stage did

This is the number set that should go straight into the deck / notebook writeup — it makes the
reduction auditable rather than a black box.

In [10]:
n_raw_rows = len(raw)
n_unique_stories = raw["story_id"].nunique()
n_stage_a = int(story_text["keyword_matched"].sum())
n_stage_b_rescued = int(story_text["stage_b_rescued"].sum())
n_final_stories = len(corpus)
n_final_raw_row_equivalent = int(corpus["mention_count"].sum())

n_india_before = (raw["corridor_id"] == "india").sum()
n_india_after = (corpus["corridor_id"] == "india").sum()

funnel = pd.DataFrame(
    {
        "stage": [
            "raw rows (all files)",
            "unique stories (cross-file dedup)",
            "stage A: keyword-matched stories",
            "stage B: additional semantic-rescued stories",
            "final relevant stories (A ∪ B)",
            "final relevant, raw-row equivalent (sum of mention_count)",
        ],
        "count": [
            n_raw_rows,
            n_unique_stories,
            n_stage_a,
            n_stage_b_rescued,
            n_final_stories,
            n_final_raw_row_equivalent,
        ],
    }
)
print(funnel.to_string(index=False))
print()
print(f"'india' corridor rows before filtering: {n_india_before:,}")
print(f"'india' corridor stories after filtering + reassignment: {n_india_after:,}")
print(f"  (this is the number that should have visibly shrunk relative to its raw share)")


                                                    stage   count
                                     raw rows (all files) 4863286
                        unique stories (cross-file dedup) 1520843
                         stage A: keyword-matched stories  141735
             stage B: additional semantic-rescued stories   10187
                           final relevant stories (A ∪ B)  151922
final relevant, raw-row equivalent (sum of mention_count)  513030

'india' corridor rows before filtering: 4,055,751
'india' corridor stories after filtering + reassignment: 73,178
  (this is the number that should have visibly shrunk relative to its raw share)


## 9. Export — CSV (flattened) and JSON (full fidelity)

CSV keeps `locations`/`entities`/`keyword_hits` as JSON-encoded strings (for spreadsheet/pandas
readability); the JSON export keeps them as native nested structures (for direct use by the
embedding/RAG step or the quant sub-score step next).

In [11]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_export = corpus.copy()
for col in ["entities", "keyword_hits", "locations"]:
    csv_export[col] = csv_export[col].apply(json.dumps)
csv_export.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(csv_export):,} rows to {OUTPUT_CSV}")

json_export = corpus.to_dict(orient="records")
with open(OUTPUT_JSON, "w") as f:
    json.dump(json_export, f, indent=2, default=str)
print(f"Wrote {len(json_export):,} records to {OUTPUT_JSON}")


Wrote 151,922 rows to D:\python time\ET\code\risk_agent\csv_backfill\gdelt\outputs/gdelt_relevant_corpus.csv
Wrote 151,922 records to D:\python time\ET\code\risk_agent\csv_backfill\gdelt\outputs/gdelt_relevant_corpus.json


## 10. Spot-check before trusting this output

Before wiring this into the RAG/quant steps, manually eyeball a sample of both the *kept* and
*dropped* stories — this is the fastest way to catch a bad threshold or a keyword list gap.

In [12]:
print("=== Sample of KEPT stories (should look genuinely energy/shipping-relevant) ===")
display_cols = ["canonical_url", "corridor_id", "mention_count", "relevance_reason"]
print(corpus[display_cols].sample(min(10, len(corpus)), random_state=42).to_string(index=False))

print("\n=== Sample of DROPPED stories (should look like genuine noise) ===")
dropped_ids = story_text[~story_text["is_relevant"]]["story_id"]
dropped_sample = raw[raw["story_id"].isin(dropped_ids)].drop_duplicates("story_id").sample(
    min(10, dropped_ids.shape[0]), random_state=42
)
print(dropped_sample[["canonical_url", "corridor_id"]].to_string(index=False))


=== Sample of KEPT stories (should look genuinely energy/shipping-relevant) ===
                                                                                                                          canonical_url  corridor_id  mention_count relevance_reason
//democracynow.org/2026/3/16/headlines/us_israeli_war_on_iran_enters_third_week_as_iranian_foreign_minister_denies_asking_for_ceasefire persian_gulf              2    keyword_match
                                                                             //lbc.co.uk/article/trump-strait-hormuz-iran-war-5HjdXc9_2       hormuz              1    keyword_match
                                           //shippingtribune.com/news/shipping/Qatar+Asks+Vessels+at+Key+LNG+Port+To+Go+Dark+for+Safety persian_gulf              3    keyword_match
                                                                     //wrp.org.uk/news/the-era-of-one-sided-deals-is-over-iran-warns-us persian_gulf              3    keyword_match
               